# Module 3, Part 1 — Why convolution

This notebook is the runnable companion to Part 1 of the Module 03 learning
module. It walks through, in code, the same ideas the module covers in
prose and diagrams:

1. Why a fully connected layer is a poor fit for images.
2. The convolution operation itself, reproducing the worked example from
   the module (same 5x5 input, same 3x3 kernel, same answer: 8).
3. Padding and stride, and how each one changes the output shape.
4. Max pooling.
5. The growth of the receptive field as convolutional layers stack up.

Run the cells in order. Nothing here needs a GPU.

In [9]:
import torch
import torch.nn as nn

torch.manual_seed(0)
print("PyTorch version:", torch.__version__)

PyTorch version: 2.10.0+cpu


## 1. Why not just flatten the image?

A fully connected layer on a flattened image needs one weight per
(input pixel, output unit) pair. The cell below computes that count for a
224x224 RGB image and a modest 512-unit hidden layer, the same numbers used
in the module.

In [10]:
h, w, c = 224, 224, 3
hidden_units = 512

flat_input_size = h * w * c
fc_params = flat_input_size * hidden_units

print(f"Flattened input size: {flat_input_size:,}")
print(f"Fully connected layer parameters (weights only): {fc_params:,}")

# Compare: one 3x3 convolutional layer with 16 output channels on the same
# 3-channel input.
conv_params = 3 * 3 * c * 16 + 16  # + 16 for the biases
print(f"\nA single 3x3 conv layer (3 in, 16 out) parameters: {conv_params:,}")
print(f"That is {fc_params / conv_params:,.0f}x fewer parameters,")
print("and the conv layer's weights are reused at every spatial position.")

Flattened input size: 150,528
Fully connected layer parameters (weights only): 77,070,336

A single 3x3 conv layer (3 in, 16 out) parameters: 448
That is 172,032x fewer parameters,
and the conv layer's weights are reused at every spatial position.


## 2. The convolution operation

This reproduces the worked example from the module exactly: a 5x5 input,
a 3x3 kernel, stride 1, no padding. We compute the full 3x3 output using
`nn.Conv2d` with the kernel hard-coded as its weight, and separately verify
the first output value (8) by hand with plain tensor arithmetic, so you can
see both the convenient API and the arithmetic it is doing underneath.

In [11]:
input_grid = torch.tensor([
    [1., 0., 1., 2., 1.],
    [0., 2., 1., 0., 1.],
    [1., 1., 3., 1., 0.],
    [0., 1., 1., 2., 1.],
    [2., 0., 1., 1., 0.],
])

kernel = torch.tensor([
    [1., 0., 1.],
    [0., 1., 0.],
    [1., 0., 1.],
])

# nn.Conv2d expects (batch, channels, height, width).
x = input_grid.unsqueeze(0).unsqueeze(0)  # -> (1, 1, 5, 5)

conv = nn.Conv2d(in_channels=1, out_channels=1, kernel_size=3, stride=1, padding=0, bias=False)
with torch.no_grad():
    conv.weight.copy_(kernel.unsqueeze(0).unsqueeze(0))

output = conv(x)
print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print("\nFull 3x3 output feature map:")
print(output.squeeze().detach())

Input shape:  torch.Size([1, 1, 5, 5])
Output shape: torch.Size([1, 1, 3, 3])

Full 3x3 output feature map:
tensor([[8., 5., 5.],
        [3., 8., 5.],
        [8., 4., 6.]])


In [12]:
# Verify the top-left output value (8) by hand: elementwise multiply the
# first 3x3 patch of the input by the kernel, then sum everything.
patch = input_grid[0:3, 0:3]
by_hand = (patch * kernel).sum()

print("Top-left 3x3 patch of the input:")
print(patch)
print("\nElementwise product with the kernel:")
print(patch * kernel)
print(f"\nSum = {by_hand.item():.0f}")
assert by_hand.item() == output[0, 0, 0, 0].item(), "hand calculation should match Conv2d's output"
print("Matches Conv2d's own output for that position: check passed.")

Top-left 3x3 patch of the input:
tensor([[1., 0., 1.],
        [0., 2., 1.],
        [1., 1., 3.]])

Elementwise product with the kernel:
tensor([[1., 0., 1.],
        [0., 2., 0.],
        [1., 0., 3.]])

Sum = 8
Matches Conv2d's own output for that position: check passed.


## 3. Padding and stride

Three Conv2d layers, same kernel size, different padding and stride. Watch
how each setting changes the output shape, confirming the rules from the
module:

- `padding=1, stride=1` keeps height and width unchanged ("same" padding
  for a 3x3 kernel).
- `padding=0, stride=1` shrinks the output slightly (no padding).
- `padding=1, stride=2` roughly halves the output (downsampling).

In [13]:
configs = [
    ("padding=1, stride=1 (same padding)", dict(padding=1, stride=1)),
    ("padding=0, stride=1 (no padding)",   dict(padding=0, stride=1)),
    ("padding=1, stride=2 (downsample)",   dict(padding=1, stride=2)),
]

x = torch.randn(1, 3, 32, 32)  # a stand-in 32x32 RGB image
print(f"Input shape: {tuple(x.shape)}\n")

for label, kwargs in configs:
    layer = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, **kwargs)
    y = layer(x)
    print(f"{label:38s} -> output shape {tuple(y.shape)}")

Input shape: (1, 3, 32, 32)

padding=1, stride=1 (same padding)     -> output shape (1, 8, 32, 32)
padding=0, stride=1 (no padding)       -> output shape (1, 8, 30, 30)
padding=1, stride=2 (downsample)       -> output shape (1, 8, 16, 16)


### Try it, in code: watch the spatial size shrink

This is the module's "Try it" exercise, worked out here instead of left to
you to write from scratch. A stack of three `nn.Conv2d` layers
(`stride=1, 2, 1`), followed by `nn.AdaptiveAvgPool2d(1)`.

In [14]:
stack = nn.Sequential(
    nn.Conv2d(3, 16, kernel_size=3, padding=1, stride=1),
    nn.Conv2d(16, 32, kernel_size=3, padding=1, stride=2),
    nn.Conv2d(32, 32, kernel_size=3, padding=1, stride=1),
)
pool = nn.AdaptiveAvgPool2d(1)

x = torch.randn(1, 3, 64, 64)
print(f"input                          {tuple(x.shape)}")

h = x
for i, layer in enumerate(stack):
    h = layer(h)
    print(f"after conv {i} (stride={layer.stride[0]})        {tuple(h.shape)}")

pooled = pool(h)
print(f"after AdaptiveAvgPool2d(1)     {tuple(pooled.shape)}")

assert h.shape[2] == 32 and h.shape[3] == 32, "only the stride-2 layer should have changed height/width"
assert pooled.shape[2] == 1 and pooled.shape[3] == 1
print("\nOnly the stride-2 layer changed the spatial size; the pool collapsed it to 1x1 regardless.")

input                          (1, 3, 64, 64)
after conv 0 (stride=1)        (1, 16, 64, 64)
after conv 1 (stride=2)        (1, 32, 32, 32)
after conv 2 (stride=1)        (1, 32, 32, 32)
after AdaptiveAvgPool2d(1)     (1, 32, 1, 1)

Only the stride-2 layer changed the spatial size; the pool collapsed it to 1x1 regardless.


## 4. Max pooling

Reproduces the module's pooling diagram: a 4x4 input, a 2x2 max-pooling
window, producing a 2x2 output. We print which value each window picked,
to make the "keeps only the largest value" rule concrete.

In [15]:
pool_input = torch.tensor([[
    [1., 3., 2., 4.],
    [5., 2., 0., 1.],
    [1., 0., 3., 2.],
    [2., 1., 1., 4.],
]]).unsqueeze(0)  # (1, 1, 4, 4)

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
pooled = maxpool(pool_input)

print("Input:")
print(pool_input.squeeze())
print("\nMax-pooled (2x2):")
print(pooled.squeeze())

# Show which value in each window was picked.
windows = {
    "top-left":     pool_input[0, 0, 0:2, 0:2],
    "top-right":    pool_input[0, 0, 0:2, 2:4],
    "bottom-left":  pool_input[0, 0, 2:4, 0:2],
    "bottom-right": pool_input[0, 0, 2:4, 2:4],
}
for name, w in windows.items():
    print(f"{name:13s} window max = {w.max().item():.0f}")

Input:
tensor([[1., 3., 2., 4.],
        [5., 2., 0., 1.],
        [1., 0., 3., 2.],
        [2., 1., 1., 4.]])

Max-pooled (2x2):
tensor([[5., 4.],
        [2., 4.]])
top-left      window max = 5
top-right     window max = 4
bottom-left   window max = 2
bottom-right  window max = 4


## 5. The growth of the receptive field

Stacking 3x3 convolutions grows the receptive field: after two stacked 3x3
layers, one output unit depends on a 5x5 region of the original input;
after three, a 7x7 region. We confirm this empirically rather than just by
formula, using a gradient trick: set every input pixel to require a
gradient, take a single output unit as the loss, call `.backward()`, and
see which input pixels ended up with a nonzero gradient. Those are exactly
the pixels that output unit's value depended on.

In [16]:
def receptive_field_size(num_layers, kernel_size=3, input_size=21):
    """Returns the side length of the receptive field of one central
    output unit after `num_layers` stacked conv layers (stride 1, no
    padding, so the output shrinks and stays centred)."""
    x = torch.zeros(1, 1, input_size, input_size, requires_grad=True)

    h = x
    for _ in range(num_layers):
        conv = nn.Conv2d(1, 1, kernel_size=kernel_size, bias=False)
        with torch.no_grad():
            conv.weight.fill_(1.0)
        h = conv(h)

    # Take the single centre output unit as the "loss" and backpropagate.
    center = h.shape[-1] // 2
    loss = h[0, 0, center, center]
    loss.backward()

    # Count how many input pixels received a nonzero gradient.
    nonzero_rows = (x.grad.abs().sum(dim=(0, 1, 3)) > 0).sum().item()
    return nonzero_rows


for n in [1, 2, 3, 4]:
    size = receptive_field_size(num_layers=n)
    print(f"{n} stacked 3x3 conv layer(s): receptive field is {size}x{size}")

1 stacked 3x3 conv layer(s): receptive field is 3x3
2 stacked 3x3 conv layer(s): receptive field is 5x5
3 stacked 3x3 conv layer(s): receptive field is 7x7
4 stacked 3x3 conv layer(s): receptive field is 9x9


Each additional 3x3 layer (stride 1, no padding) grows the receptive
field by 2 in each dimension, exactly matching the module: 3x3, then 5x5,
then 7x7, then 9x9. This is also why downsampling (stride or pooling)
accelerates the growth further: a stride-2 step doubles the effective
reach of every layer that follows it, which the module revisits in Part 2
when discussing why deep networks can still see small objects despite very
large receptive fields overall.

## Summary

- A convolution's weight sharing cuts parameter count dramatically
  compared to a fully connected layer on the same image, and lets a learned
  pattern transfer across spatial positions.
- Padding and stride directly control the output's spatial size; the
  formulas from the module match what `nn.Conv2d` actually produces.
- Max pooling reduces spatial size without learning any weights, by
  keeping only the largest value in each window.
- Stacking convolutional layers grows the receptive field, confirmed here
  empirically rather than just asserted.